# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We begin by loading the Croissant metadata from the FAIR^2 dataset into Python using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and access its information
dataset = mlc.Dataset(croissant_url)

# Access dataset-level information
print(f"Loaded dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {dataset.metadata.keywords}\n")
if hasattr(dataset.metadata, 'spatialCoverage'):
    print(f"Coverage: {dataset.metadata.spatialCoverage}\n")


## 2. Data Overview

Let's inspect the available record sets, their fields, and column IDs. In Croissant, every data entity (record set, field, column) is referenced by a unique `@id`. We'll enumerate all record sets and fields by their IDs for transparency and reproducibility.

In [ ]:
# List the record set @ids in the dataset
print("Record Sets in the Dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'Unnamed RecordSet')}")

# For each record set, print all field @ids
print("\nAvailable fields (by @id) for each record set:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        if isinstance(f, str):
            print(f"  - field @id: {f}")
        elif isinstance(f, dict) and '@id' in f:
            print(f"  - field @id: {f['@id']}")

## 3. Data Extraction

Let's extract data from every available record set using their `@id`. We will store each table as a `pandas.DataFrame`, keyed by its record set `@id`.

In [ ]:
# Build a list of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Extract data for each record set (by @id). This may take time if network or files are large.
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {record_set_id}")
        else:
            print(f"No records found for {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

print("\nAvailable DataFrame columns for each record set:")
for rsid, df in dataframes.items():
    print(f"- RecordSet {rsid}: columns -> {df.columns.tolist()}")

# If at least one dataframe loaded, display a preview from the first one
if dataframes:
    first_key = list(dataframes.keys())[0]
    print(f"\nPreview of data for RecordSet {first_key}:")
    display(dataframes[first_key].head())

## 4. Exploratory Data Analysis (EDA)

Suppose our example dataset has a record set containing regression coefficients and associated statistics (`@id` and columns per previous step). We'll filter, normalize, and group by a field, referencing columns using their `@id` as required. Adjust the variables as needed for your exploration.

In [ ]:
# Choose a record set for analysis. Replace the below with the appropriate @id from your data.
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Selected RecordSet: {record_set_id}\nColumns: {df.columns.tolist()}")

    # Pick a numeric field @id. For this example, we'll try to pick a likely numeric column.
    possible_numeric_ids = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int, np.float32, np.int32]]
    if not possible_numeric_ids:
        # Try to pick a float-like column by attempted conversion
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                if df[c].dtype in [np.float64, np.int64, float, int, np.float32, np.int32]:
                    possible_numeric_ids.append(c)
            except Exception:
                continue
    if possible_numeric_ids:
        numeric_field_id = possible_numeric_ids[0]
    else:
        # Fallback dummy
        numeric_field_id = df.columns[0]

    print(f"\nNumeric field selected for EDA: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field (try to pick a categorical column)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping by field '@id': {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found to demonstrate grouping.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization

Visualize distributions or relationships using matplotlib/seaborn. Below, we'll plot the distribution of the chosen numeric field and, if grouping was possible, a mean comparison by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion

- This notebook demonstrated how to load, explore, and process a Croissant dataset with `mlcroissant` using only `@id` references for each entity.
- We reviewed the available record sets and fields, loaded them as DataFrames, and performed EDA including filtering, normalization, and grouping.
- Visualizations gave further insight into the distributions and potential group effects in the dataset.

For further analysis, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and adapt the exploration for your specific use case.